# Causal LM Loss、PPL 与 BPB：客服语料评测实战

**面试问题：next-token loss、padding/文档边界 mask、PPL 和 BPB 怎样计算才可比较？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

团队要比较两个 tokenizer 下同一批中文客服回复的语言模型质量。三条回复长度不同，其中一条被 padding；训练时还会把多文档 packing 到同一序列。案例直接使用每个有效 token 的负对数似然，展示错误的 batch 平均为什么会偏向短样本，并用每字节比特数避免 tokenizer 粒度造成的假优势。

### 输入预览：长度不同的客服回复

In [1]:
import math  # 导入指数和对数函数以计算 PPL 与 BPB。
from pprint import pprint  # 导入结构化打印工具展示逐样本字段。

samples = [  # 构造三条具有可读语义和不同长度的客服回复评分记录。
    {"text": "已为您退款", "token_nll": [0.31, 0.42, 0.38, 0.29, 0.35, 0.40], "bytes": len("已为您退款".encode("utf-8"))},  # 短回复具有六个有效预测位置。
    {"text": "订单已发出，预计明天送达", "token_nll": [0.28, 0.33, 0.40, 0.47, 0.51, 0.36, 0.41, 0.32, 0.39, 0.44, 0.30], "bytes": len("订单已发出，预计明天送达".encode("utf-8"))},  # 中等回复包含更多配送信息。
    {"text": "请提供订单号和支付截图，我们会在两个工作日内核实", "token_nll": [0.62, 0.55, 0.48, 0.51, 0.46, 0.58, 0.53, 0.49, 0.57, 0.45, 0.50, 0.54, 0.47, 0.52], "bytes": len("请提供订单号和支付截图，我们会在两个工作日内核实".encode("utf-8"))},  # 长回复更困难且不能被短样本平均稀释。
]  # 完成评测样本列表。
print("评测样本及有效 token 数：")  # 输出输入预览标题。
for sample in samples:  # 逐条展示可读文本、token 数和 UTF-8 字节数。
    print(f'{sample["text"]} | tokens={len(sample["token_nll"]):2d} | bytes={sample["bytes"]:2d}')  # 显示 PPL 与 BPB 的两个不同分母。

评测样本及有效 token 数：
已为您退款 | tokens= 6 | bytes=15
订单已发出，预计明天送达 | tokens=11 | bytes=36
请提供订单号和支付截图，我们会在两个工作日内核实 | tokens=14 | bytes=72


## Baseline 基线：先算每条 PPL 再做等权平均

In [2]:
def sequence_ppl(token_nll):  # 定义单条序列的标准 perplexity 计算函数。
    return math.exp(sum(token_nll) / len(token_nll))  # 对有效 token 平均 NLL 后取指数。

per_sample_ppl = [sequence_ppl(sample["token_nll"]) for sample in samples]  # 分别计算三条回复的 PPL。
wrong_batch_ppl = sum(per_sample_ppl) / len(per_sample_ppl)  # 构造常见但错误的样本等权平均基线。
print("逐样本 PPL：")  # 输出每条回复的难度供错误分析。
for sample, ppl in zip(samples, per_sample_ppl):  # 对齐展示文本和对应 PPL。
    print(f'{ppl:6.3f} | {sample["text"]}')  # 让学习者看到长难样本的 PPL 更高。
print(f"错误的 batch 等权 PPL = {wrong_batch_ppl:.4f}")  # 展示错误聚合结果作为后续对照。

逐样本 PPL：
 1.431 | 已为您退款
 1.466 | 订单已发出，预计明天送达
 1.681 | 请提供订单号和支付截图，我们会在两个工作日内核实
错误的 batch 等权 PPL = 1.5260


### 核心实现：按有效 token 汇总分子与分母

In [3]:
def corpus_metrics(records):  # 实现跨 batch 和跨设备都可复用的语料级聚合。
    total_nll = sum(sum(record["token_nll"]) for record in records)  # 汇总所有有效预测位置的负对数似然。
    total_tokens = sum(len(record["token_nll"]) for record in records)  # 汇总与 NLL 对齐的有效 token 数。
    total_bytes = sum(record["bytes"] for record in records)  # 汇总原始 UTF-8 文本字节数以计算 BPB。
    mean_nll = total_nll / total_tokens  # 使用全局 token 分母得到无 batch 偏差的平均损失。
    return {"nll": mean_nll, "ppl": math.exp(mean_nll), "bpb": total_nll / (total_bytes * math.log(2)), "tokens": total_tokens, "bytes": total_bytes}  # 返回可审计的分子分母和派生指标。

correct = corpus_metrics(samples)  # 对三条不同长度回复计算正确的语料级指标。
print("正确的 token 加权聚合：", correct)  # 展示平均 NLL、PPL、BPB 和真实分母。
print(f"错误等权与正确聚合的 PPL 差值 = {wrong_batch_ppl - correct['ppl']:+.5f}")  # 量化短样本等权造成的偏差。

正确的 token 加权聚合： {'nll': 0.4396774193548387, 'ppl': 1.5522064259923716, 'bpb': 0.1598693772952567, 'tokens': 31, 'bytes': 123}
错误等权与正确聚合的 PPL 差值 = -0.02619


### Packing 与 padding：哪些位置不能计入 loss

In [4]:
packed_tokens = ["<BOS>", "已", "退", "款", "<EOS>", "<BOS>", "订", "单", "已", "发", "出", "<PAD>", "<PAD>"]  # 构造包含两个文档和 padding 的教学序列。
packed_nll = [0.0, 0.30, 0.41, 0.35, 0.29, 0.0, 0.28, 0.33, 0.40, 0.47, 0.36, 8.0, 8.0]  # 给 padding 故意放入巨大损失以暴露错误 mask。
loss_mask = [0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0]  # 屏蔽 BOS、文档新起点和 padding 位置。
effective_nll = [loss * mask for loss, mask in zip(packed_nll, loss_mask)]  # 只保留真正参与 next-token 目标的位置。
masked_mean = sum(effective_nll) / sum(loss_mask)  # 用有效位置数量而不是序列长度作为分母。
unmasked_mean = sum(packed_nll) / len(packed_nll)  # 构造把 padding 和边界算入 loss 的错误结果。
print("packed token / mask / NLL：")  # 输出逐位置观察标题。
for token, mask, loss in zip(packed_tokens, loss_mask, packed_nll):  # 逐 token 展示哪些位置参与训练。
    print(f'{token:6} mask={mask} nll={loss:4.2f}')  # 直观看到 BOS 与 PAD 被正确排除。
print(f"正确 masked NLL={masked_mean:.4f}，错误 unmasked NLL={unmasked_mean:.4f}")  # 展示错误 mask 对指标的巨大污染。

packed token / mask / NLL：
<BOS>  mask=0 nll=0.00
已      mask=1 nll=0.30
退      mask=1 nll=0.41
款      mask=1 nll=0.35
<EOS>  mask=1 nll=0.29
<BOS>  mask=0 nll=0.00
订      mask=1 nll=0.28
单      mask=1 nll=0.33
已      mask=1 nll=0.40
发      mask=1 nll=0.47
出      mask=1 nll=0.36
<PAD>  mask=0 nll=8.00
<PAD>  mask=0 nll=8.00
正确 masked NLL=0.3544，错误 unmasked NLL=1.4762


## 结果解读：为什么跨 tokenizer 要看 BPB

In [5]:
tokenizer_a = [{"text": row["text"], "token_nll": row["token_nll"], "bytes": row["bytes"]} for row in samples]  # 把当前细粒度 tokenizer 结果记为方案 A。
tokenizer_b = [{"text": row["text"], "token_nll": [value * 1.72 for value in row["token_nll"][::2]], "bytes": row["bytes"]} for row in samples]  # 模拟更粗 tokenizer：token 更少但单 token NLL 更大。
metrics_a = corpus_metrics(tokenizer_a)  # 计算细粒度 tokenizer 的 PPL 与 BPB。
metrics_b = corpus_metrics(tokenizer_b)  # 计算粗粒度 tokenizer 的 PPL 与 BPB。
print("跨 tokenizer 对比：")  # 输出可比较指标表标题。
print(f"A 细粒度 | tokens={metrics_a['tokens']:2d} | PPL={metrics_a['ppl']:.3f} | BPB={metrics_a['bpb']:.3f}")  # 展示细粒度方案的 token 指标和字节指标。
print(f"B 粗粒度 | tokens={metrics_b['tokens']:2d} | PPL={metrics_b['ppl']:.3f} | BPB={metrics_b['bpb']:.3f}")  # 展示粗粒度方案不能直接用 PPL 与 A 比较。
print("解读：token 粒度变化会改变 PPL 的基本单位，BPB 才把两者还原到同一原始字节分母。")  # 明确回答面试中的跨 tokenizer 比较陷阱。

跨 tokenizer 对比：
A 细粒度 | tokens=31 | PPL=1.552 | BPB=0.160
B 粗粒度 | tokens=16 | PPL=2.113 | BPB=0.140
解读：token 粒度变化会改变 PPL 的基本单位，BPB 才把两者还原到同一原始字节分母。


## 失败案例：滑窗重叠 token 被重复计分

In [6]:
long_nll = [0.22, 0.31, 0.29, 0.35, 0.42, 0.38, 0.33, 0.40, 0.36, 0.30]  # 构造需要两个重叠窗口评分的长回复。
window_one = list(range(0, 6))  # 第一个窗口负责位置零到五。
window_two = list(range(4, 10))  # 第二个窗口为保留上下文而与前窗重叠两个位置。
wrong_positions = window_one + window_two  # 错误实现会把重叠位置四和五重复加入分子分母。
correct_positions = window_one + [index for index in window_two if index > window_one[-1]]  # 正确实现只评分第二窗的新 token。
wrong_sliding_nll = sum(long_nll[index] for index in wrong_positions) / len(wrong_positions)  # 计算重复计分后的错误均值。
correct_sliding_nll = sum(long_nll[index] for index in correct_positions) / len(correct_positions)  # 计算每个 token 恰好一次的正确均值。
print("错误评分位置：", wrong_positions)  # 展示重复索引帮助定位滑窗 bug。
print("正确评分位置：", correct_positions)  # 展示 overlap 只提供上下文而不重复记账。
print(f"错误滑窗 NLL={wrong_sliding_nll:.4f}，正确 NLL={correct_sliding_nll:.4f}")  # 量化重复评分造成的指标偏移。

错误评分位置： [0, 1, 2, 3, 4, 5, 4, 5, 6, 7, 8, 9]
正确评分位置： [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
错误滑窗 NLL=0.3467，正确 NLL=0.3360


### 生产边界

In [7]:
audit_record = {"nll_sum": round(correct["nll"] * correct["tokens"], 6), "valid_tokens": correct["tokens"], "source_bytes": correct["bytes"], "template_version": "support-v3", "tokenizer_version": "tok-a-2026-07"}  # 构造分布式汇总和版本复现需要的审计记录。
print("评测审计记录：", audit_record)  # 展示生产系统不应只保存一个 PPL 浮点数。
print("生产替换点：各 rank 只 all-reduce NLL 总和与有效 token 总数，并冻结 tokenizer、chat template、窗口策略和数据快照。")  # 说明多卡评测与版本合同。

评测审计记录： {'nll_sum': 13.63, 'valid_tokens': 31, 'source_bytes': 123, 'template_version': 'support-v3', 'tokenizer_version': 'tok-a-2026-07'}
生产替换点：各 rank 只 all-reduce NLL 总和与有效 token 总数，并冻结 tokenizer、chat template、窗口策略和数据快照。


## 回归测试：只保护分母、mask 与覆盖范围

In [8]:
assert correct["tokens"] == sum(len(row["token_nll"]) for row in samples)  # 验证语料聚合没有漏掉或重复有效 token。
assert correct["ppl"] != wrong_batch_ppl  # 验证不同长度样本下没有退化为样本等权平均。
assert masked_mean < unmasked_mean  # 验证巨大 padding 损失被 mask 正确排除。
assert correct_positions == list(range(len(long_nll)))  # 验证滑窗评分恰好覆盖每个 token 一次。
assert audit_record["valid_tokens"] > 0  # 验证审计记录保存了可解释的真实分母。
print("回归测试通过：全局分母、边界 mask、滑窗覆盖和审计字段均正确。")  # 显示少量测试真正保护的合同。

回归测试通过：全局分母、边界 mask、滑窗覆盖和审计字段均正确。
